# Compare v1.0 vs v1.1, full FDSI benchmark

RoA and SIL over all 100 recordings for every applied config of both versions (all `lr_fixed`):

| branch | v1.0 (`../fdsi_benchmark/`) | v1.1 (this folder) |
|---|---|---|
| fixed | no adaptation, from sample 0 | no adaptation, from the calibration end |
| sv_loss | `tpe_sv_median` | `mtpe_sv_mean` |
| pareto min-sv | `tpe_pareto` | `mtpe_pareto_min_sv` |
| pareto knee | -- (compared with v1.0 pareto min-sv) | `mtpe_pareto_knee` |
| roa | `tpe_roa` | `mtpe_roa` |

Pure disk read: no decomposition or search runs here.

**How to read it.** A v1.1 vs v1.0 difference mixes three changes:
1. **The search itself:** sampler, `centroid_momentum`, per-unit `sv_loss`, unit selection.
2. **Where adaptation starts:** the calibration end, with CBSS's output kept over the window.
3. **Which units drive the search:** CoV-ISI selection.

Three views help separate them:
- **The fixed-baseline pair** (no search involved) isolates (2).
- **Post-calibration-only RoA** removes the calibration window from both versions.
- **The search diagnostics** show how (1) and (3) moved the chosen parameters.

## Config

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '../..')          # package root
sys.path.insert(0, str(Path.cwd()))  # this folder, for fdsi_v11 (which imports ../fdsi_benchmark/fdsi_common)

import fdsi_v11 as fv
fc = fv.fc

g = fv.Grid.from_yaml(Path('../../configs/data_configs/fdsi_benchmark_grid.yaml'))
BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')   # v1.0's base config

print(f'{len(g.recordings())} recordings; calibration window [0, {g.cal_end}) samples; '
      f'pool {g.pool_conditions} ({g.optim_sub}, {g.optim_snr} dB); holdout {g.holdout_conditions}')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from adapt_decomp.utils.plots import plot_metric_heatmap, plot_phase_bar

sns.set_theme(style='whitegrid', font_scale=1.1)

BRANCHES = ['sv_loss', 'pareto_min_sv', 'pareto_knee', 'roa']
CONFIGS = fv.configs(*BRANCHES)
CONFIG_ORDER = [label for label, *_ in CONFIGS]
PAIRS = fv.pairs(*BRANCHES)
PALETTE = {label: ('#888888' if label.startswith('fixed') else '#4C72B0' if version == 'v1.0' else '#DD8452')
           for label, version, *_ in CONFIGS}

## Aggregate from disk

In [ ]:
df_roa = fv.roa_table(g, CONFIGS)
df_sil = fv.sil_table(g, CONFIGS)
df_phase = fv.phase_table(g, CONFIGS)
df_post = fv.window_roa_table(g, CONFIGS, slice(g.cal_end, None))
configs_present = [c for c in CONFIG_ORDER if c in set(df_roa['config'])]
for d in (df_roa, df_post):
    d['set'] = np.where(d['condition'].isin(g.pool_conditions), 'pool', 'holdout')
print(f'{len(df_roa)} unit rows; configs present: {configs_present}')

## 1. Overall summary

In [ ]:
def overall(df, value='roa_pct', threshold=90):
    table = df.groupby('config')[value].agg(
        n_unit_rows='count', mean='mean', median='median',
        pct_ge_threshold=lambda x: (x >= threshold).mean() * 100)
    return table.reindex(configs_present).round(2)

overall_roa = overall(df_roa)
overall_roa

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
overall_roa['mean'].plot.bar(ax=ax, color=[PALETTE[c] for c in overall_roa.index])
ax.set(ylabel='Mean RoA (%)', xlabel='', title='Mean RoA by config (grey: fixed, blue: v1.0, orange: v1.1)')
ax.tick_params(axis='x', rotation=35)
for label in ax.get_xticklabels():
    label.set_ha('right')
plt.tight_layout()
plt.show()

## 2. Paired per-unit change, v1.1 vs v1.0

Each unit's RoA under the v1.1 config minus under its v1.0 counterpart. Wilcoxon signed-rank
test on the paired differences; units within 0.5 RoA points count as unchanged.

In [ ]:
paired = fv.paired_summary(df_roa, PAIRS)
paired.round(4)

In [ ]:
deltas = pd.concat([fv.paired_delta(df_roa, before, after).assign(pair=f'{after}\nvs {before}')
                    for before, after in PAIRS])
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=deltas, x='pair', y='delta', flierprops=dict(marker='.', markersize=3, alpha=0.4), ax=ax)
ax.axhline(0, color='k', lw=1)
ax.set(xlabel='', ylabel='RoA change, v1.1 - v1.0 (points)', title='Per-unit RoA change')
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, len(PAIRS), figsize=(4.5 * len(PAIRS), 4), sharey=True)
for ax, (before, after) in zip(np.atleast_1d(axs), PAIRS):
    grid_delta = (fv.paired_delta(df_roa, before, after)
                  .groupby(['condition', 'snr'])['delta'].mean().unstack('snr')
                  .reindex(index=g.conditions, columns=g.snr_levels))
    sns.heatmap(grid_delta, annot=True, fmt='.1f', cmap='RdBu', center=0, ax=ax, cbar=ax is axs[-1])
    ax.set(title=f'{after}\nvs {before}', xlabel='SNR (dB)', ylabel='')
plt.suptitle('Mean per-unit RoA change by condition x SNR')
plt.tight_layout()
plt.show()

## 3. Pool vs holdout conditions

The searches only saw the pooled conditions (sub-01, 30 dB); holdout conditions test whether
the chosen configs generalise.

In [ ]:
by_set = (df_roa.groupby(['set', 'config'])['roa_pct'].mean().unstack('config')
          .reindex(columns=configs_present).round(2))
paired_by_set = pd.concat({s: fv.paired_summary(df_roa[df_roa['set'] == s], PAIRS)
                           for s in ('pool', 'holdout')})
display(by_set)
paired_by_set[['n_units', 'mean_delta', 'pct_improved', 'pct_worse', 'wilcoxon_p']].round(4)

## 4. Phase RoA -- triangular contractions

First isometric hold (calibration window included), ramp, last hold. Adapting from the
calibration end and tracking faster (`centroid_momentum`) should show up mostly on the ramp.

In [ ]:
plot_phase_bar(df_phase, configs_present, g.triangular, g.snr_levels, fc.PHASE_ORDER, fc.PHASE_LABELS)
plt.suptitle('Phase RoA -- triangular contractions', fontsize=13)
plt.show()

In [ ]:
phase_paired = pd.concat({phase: fv.paired_summary(df_phase[df_phase['phase'] == phase], PAIRS)
                          for phase in fc.PHASE_ORDER})
phase_paired[['n_units', 'mean_delta', 'pct_improved', 'pct_worse', 'wilcoxon_p']].round(4)

## 5. After the calibration window only

RoA over [5 s, end) for both versions. This removes v1.1's CBSS-over-window output, which
tends to raise its full-recording RoA, and v1.0's re-adaptation over the window. The fixed
pair here shows the effect of batch alignment alone.

In [ ]:
display(overall(df_post))
fv.paired_summary(df_post, PAIRS).round(4)

## 6. SIL (GT-free)

In [ ]:
display(overall(df_sil, 'sil', threshold=0.9))
fv.paired_summary(df_sil, PAIRS, value='sil', tol=0.005).round(4)

In [ ]:
plot_metric_heatmap(df_sil, 'sil', configs_present, g.conditions, g.snr_levels, agg='mean', vmin=0, vmax=1)
plt.suptitle('Mean full-recording SIL by condition x SNR', fontsize=12)
plt.show()

## 7. Chosen hyperparameters

Promoted winners of both versions (v1.0 kept `centroid_momentum` at 0.95).

In [ ]:
fv.hyperparameter_table(BRANCHES)

## 8. Search diagnostics

For each search: each pooled recording's share of the pooled `sv_loss` (unit-count bias,
v1.0 sum vs v1.1 per-unit mean), and the wall time per trial.

In [ ]:
import yaml
from adapt_decomp import load_data

with open('../../configs/data_configs/fdsi_pool_memory_example.yaml') as f:
    pool = load_data(yaml.safe_load(f))
cov_th = fv.SEARCH['unit_selection_kwargs']['cov_th']
units = {'v1.0': {n: d.calibration.sources.shape[1] for n, d in pool.items()},
         'v1.1': {n: int(d.calibration.unsupervised_mask(cov_th=cov_th).sum()) for n, d in pool.items()}}

SEARCH_PAIRS = {'sv_loss': ('tpe_sv_median', 'mtpe_sv_mean'), 'pareto': ('tpe_pareto', 'mtpe_pareto'),
                'roa': ('tpe_roa', 'mtpe_roa')}
studies = {(search, version): fv.load_study(g.opt_dir(version, d))
           for search, dirs in SEARCH_PAIRS.items() for version, d in zip(('v1.0', 'v1.1'), dirs)}

shares = pd.concat([fv.pooled_loss_shares(study, units[version]).assign(search=search, version=version)
                    for (search, version), study in studies.items() if study is not None])
(shares.groupby(['search', 'version', 'recording'])
       .agg(units=('units', 'first'), mean_share=('share', 'mean')).unstack('recording').round(3))

In [ ]:
durations = pd.DataFrame({f'{search} {version}': fv.trial_durations(study).describe()
                          for (search, version), study in studies.items() if study is not None})
durations.loc[['count', '50%', 'mean', 'max']].round(2)